In [1]:
pip install requests beautifulsoup4 Pillow pytesseract PyPDF2

  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl (7.2 MB)

   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   --- ------------------------------------  1/11 [typing-extensions]
   ------- --------------------------------  2/11 [soupsieve]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- -----------------------------  3/11 [PyPDF2]
   ---------- ----

In [ ]:
import os
import io
import requests
import pytesseract
from PIL import Image
from PyPDF2 import PdfWriter, PdfReader
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def download_and_ocr_to_pdf(base_url, output_folder, pdf_filename):
    pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    print(f"--- TAHAP 1: DOWNLOAD GAMBAR ---")
    i = 1
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    
    # Kita terus download sampai dapat 404
    while True:
        file_name = f"{i}.jpg"
        file_path = os.path.join(output_folder, file_name)
        
        if os.path.exists(file_path):
            # Opsional: Bisa tambahkan verifikasi file size di sini jika perlu
            pass
        else:
            image_url = f"{base_url}{i}.jpg"
            try:
                response = requests.get(image_url, headers=headers, verify=False, timeout=30)
                if response.status_code == 404:
                    print(f"Halaman {i} tidak ditemukan (Akhir dokumen). Berhenti mendownload.")
                    break
                response.raise_for_status()
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f"Berhasil mengunduh: {file_name}")
            except Exception as e:
                print(f"Error di halaman {i}: {e}")
                break
        i += 1
        if i > 500: break # Safety limit

    print(f"\n--- TAHAP 2: PROSES PDF & OCR ---")
    
    # Ambil semua file .jpg di folder
    all_files = [f for f in os.listdir(output_folder) if f.endswith('.jpg')]
    
    # LOGIKA PENTING: Urutkan berdasarkan angka murni
    # Tanpa ini, urutannya jadi: 1.jpg, 10.jpg, 100.jpg, 11.jpg, dst.
    all_files.sort(key=lambda x: int(os.path.basename(x).split('.')[0]))
    
    print(f"Ditemukan {len(all_files)} gambar. Menyusun secara urut...")
    
    output_pdf = PdfWriter()
    stream_refs = [] # Untuk menjaga memory agar tidak corrupt
    
    for file_name in all_files:
        file_path = os.path.join(output_folder, file_name)
        print(f"Menambahkan ke PDF: {file_name}")
        
        try:
            # 1. OCR ke PDF bytes
            # Lang 'ind' untuk Bahasa Indonesia agar teks presisi
            pdf_bytes = pytesseract.image_to_pdf_or_hocr(file_path, extension='pdf', lang='ind')
            
            # 2. Masukkan ke stream memory
            pdf_stream = io.BytesIO(pdf_bytes)
            stream_refs.append(pdf_stream) # WAJIB: Agar tidak hilang saat loop lanjut
            
            # 3. Baca stream dan tambahkan ke PDF utama
            reader = PdfReader(pdf_stream)
            output_pdf.add_page(reader.pages[0])
            
        except Exception as e:
            print(f"GAGAL memproses {file_name}: {e}")
            
    # Pastikan file tidak sedang dibuka sebelum menulis
    try:
        with open(pdf_filename, "wb") as f:
            output_pdf.write(f)
        print(f"\nSUKSES! PDF tersimpan sebagai: {pdf_filename}")
    except PermissionError:
        print(f"\nERROR: Tidak bisa menyimpan file. Pastikan '{pdf_filename}' sedang TIDAK DIBUKA di aplikasi lain.")

# CONFIG
BASE_URL = "https://ir.unair.ac.id/uploaded_files/temporary/DigitalCollection/NmM0YjE4NzU4YWYxNWM3NTEwMGI2MTQ0YjAxNTk5MjQ5YmY3NTQ1Nw==/files/mobile/"
FOLDER_NAMA = "Judul Skripsi Kating"
OUTPUT_PDF = "Judul_Skripsi_Fix_Urut.pdf"

download_and_ocr_to_pdf(BASE_URL, FOLDER_NAMA, OUTPUT_PDF)
